# v01 — Zero-shot baseline

End-to-end Colab runner for `v01_zeroshot_baseline`. All knobs (model, dtype, batch size, max tokens, ...) live in [`app/physics_solution/config.py`](../../config.py) — edit there, not here.

Runs **directly off the Drive copy** (Colab Oct-2025+ runtime is fast enough that copying to `/content/` is no longer worth the disk churn). The repo path `REPO_ROOT` below points to your Drive folder.

## 1. Mount Drive + chdir


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO_ROOT = '/content/drive/MyDrive/NCKH/Exact_2026_Laplace-s_Red_Devils'  # change if needed
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('cwd:', os.getcwd())

In [ ]:
!pip -q install -r app/physics_solution/requirements.txt

### Install Qwen3.5 fast-attention kernels (`flash-linear-attention` + `causal-conv1d`)

On Colab Oct-2025+ runtimes both wheels install cleanly. If `causal_conv1d: False` shows up, the model still runs via torch fallback at ~3× slower — accuracy is unaffected.

In [ ]:
from app.physics_solution.shared.colab.setup import install_fast_kernels
install_fast_kernels()

In [ ]:
# setup_colab handles .env + LangSmith.
from app.physics_solution.shared.colab.setup import setup_colab
setup_colab(repo_root=REPO_ROOT)

In [ ]:
!nvidia-smi -L

## 2. Test set paths

Two evaluation runs: **full test** (original data) and **golden** (DeepSeek-rewritten CoT).

| File | Rows | Scope |
|---|---|---|
| `full_test.csv` | 1352 | All answer types — original CoT |
| `deepseek-v4-pro_golden_data.csv` | 1352 | All answer types — golden CoT |

In [ ]:
import os

# --- Full test (original data) ---
FULL_TEST_FILE  = 'app/physics_solution/data/test/full_test.csv'
FULL_OUT_FILE   = 'app/physics_solution/versions/v01_zeroshot_baseline/output/results_full.json'

# --- Golden test (DeepSeek-rewritten CoT) ---
GOLDEN_TEST_FILE = 'app/physics_solution/data/golden/deepseek-v4-pro_golden_data.csv'
GOLDEN_OUT_FILE  = 'app/physics_solution/versions/v01_zeroshot_baseline/output/results_golden.json'

for label, f in [('Full', FULL_TEST_FILE), ('Golden', GOLDEN_TEST_FILE)]:
    assert os.path.exists(f), f"Not found: {f}"
    print(f'{label}: {f}')

## 3. Inference

Runs both full test and golden test sequentially. All knobs (batch size, dtype, max_new_tokens, ...) come from `config.py`.

In [ ]:
# --- 3a. Full test ---
!python -m app.physics_solution.cli.inference \
    --version v01_zeroshot_baseline \
    --test-file {FULL_TEST_FILE} \
    --out {FULL_OUT_FILE}

# --- 3b. Golden test ---
!python -m app.physics_solution.cli.inference \
    --version v01_zeroshot_baseline \
    --test-file {GOLDEN_TEST_FILE} \
    --out {GOLDEN_OUT_FILE}

## 4. Push to HF (with full metadata)


In [ ]:
!python -m app.physics_solution.cli.upload_model \
    --version-num 1 --strategy zeroshot \
    --results {FULL_OUT_FILE} \
    --test-file {FULL_TEST_FILE} \
    --extra-results {GOLDEN_OUT_FILE}

## 5. Inspect results


In [ ]:
import json, pandas as pd

for label, path in [('FULL', FULL_OUT_FILE), ('GOLDEN', GOLDEN_OUT_FILE)]:
    print(f'\n{"="*60}')
    print(f'  {label}: {path}')
    print(f'{"="*60}')
    data = json.loads(open(path, encoding='utf-8').read())
    print('summary:', json.dumps(data['summary'], indent=2, ensure_ascii=False))
    df = pd.DataFrame(data['rows'])
    wrong = df[~df['is_correct']].copy()
    wrong['reached_final'] = wrong['completion'].str.contains('FINAL ANSWER', case=False, regex=False)
    print(f"\nWrong: {len(wrong)} / {len(df)}")
    print(f"  ... never reached FINAL ANSWER: {(~wrong['reached_final']).sum()}")
    display(wrong[['id', 'gold_answer', 'pred_numeric', 'pred_unit', 'gold_unit', 'reached_final']].head(20))

## 6. Deep error analysis

Run the error analyzer on the results CSV to generate a full report with per-answer-type breakdowns:

```bash
python -m app.physics_solution.eda.scripts.v2.error_analysis \
    app/physics_solution/versions/v01_zeroshot_baseline/output/results_full.csv \
    --out docs/eda/v01_error_report_full.md
```

Or open [`app/physics_solution/eda/notebooks/v2/error_analysis.ipynb`](../../eda/notebooks/v2/error_analysis.ipynb) for interactive exploration.